In [ ]:
# in notebook or utils
import pandas as pd
JHU_CONF_URL = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
df_conf = pd.read_csv(JHU_CONF_URL)
# tidy
df = df_conf.melt(id_vars=['Province/State','Country/Region','Lat','Long'],
                  var_name='date', value_name='cum_cases')
df['date'] = pd.to_datetime(df['date'])
df = df.rename(columns={'Province/State':'state','Country/Region':'country'})


In [ ]:
def extract_region_daily(df, country, state=None):
    sel = df[df['country']==country]
    if state:
        sel = sel[sel['state']==state]
    # aggregate across provinces if needed
    daily = sel.groupby('date')['cum_cases'].sum().sort_index()
    new = daily.diff().fillna(0).clip(lower=0)   # remove negatives
    out = pd.DataFrame({'ds': daily.index, 'y': new.values, 'cases_cum': daily.values})
    return out


In [ ]:
def build_binary_regressor(dates_index, event_dates):
    s = pd.Series(0, index=dates_index)
    for d in event_dates:
        dt = pd.to_datetime(d)
        if dt in s.index: s.loc[dt] = 1
    return s

dates = pd.date_range(start='2020-01-01', end='2022-12-31')
lockdowns = ['2020-03-15','2020-11-01']
vax_starts = ['2020-12-14']
df_reg = pd.DataFrame({
    'lockdown': build_binary_regressor(dates, lockdowns),
    'vax_start': build_binary_regressor(dates, vax_starts)
})


In [ ]:
from prophet import Prophet

m = Prophet(weekly_seasonality=True, daily_seasonality=False, yearly_seasonality=False,
            changepoint_prior_scale=0.05)
# add regressors
m.add_regressor('lockdown')
m.add_regressor('vax_start')

train = df_region[['ds','y']].copy()
train = train.merge(df_reg, left_on='ds', right_index=True, how='left').fillna(0)
m.fit(train)

future = m.make_future_dataframe(periods=28)
future = future.merge(df_reg, left_on='ds', right_index=True, how='left').fillna(0)
fcst = m.predict(future)


In [ ]:
import pmdarima as pm

series = df_region.set_index('ds')['y'].asfreq('D').fillna(0)
model = pm.auto_arima(series, seasonal=True, m=7, stepwise=True, suppress_warnings=True)
model.summary()
preds, conf = model.predict(n_periods=28, return_conf_int=True)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

def create_sequences(values, seq_len=28):
    X, y = [], []
    for i in range(len(values)-seq_len):
        X.append(values[i:i+seq_len])
        y.append(values[i+seq_len])
    return np.array(X), np.array(y)

vals = series.values.reshape(-1,1)
scaler = MinMaxScaler()
vals_s = scaler.fit_transform(vals)

seq_len = 28
X, y = create_sequences(vals_s, seq_len)
# split train/test (e.g., last 28 days reserved)
split = int(0.8*len(X))
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

model = Sequential([
    LSTM(64, input_shape=(seq_len,1), return_sequences=False),
    Dropout(0.2),
    Dense(1)
])
model.compile('adam','mse')
model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_val,y_val))
# forecasting: iterative multi-step
def forecast_lstm_multi(model, last_window, steps, scaler):
    preds=[]
    window = last_window.copy()
    for _ in range(steps):
        p = model.predict(window.reshape(1,seq_len,1))[0,0]
        preds.append(p)
        window = np.roll(window, -1)
        window[-1] = p
    return scaler.inverse_transform(np.array(preds).reshape(-1,1)).ravel()


In [ ]:
import numpy as np

def rmse(y_true,y_pred): return np.sqrt(np.mean((y_true-y_pred)**2))
def mae(y_true,y_pred): return np.mean(np.abs(y_true-y_pred))
def smape(y_true,y_pred):
    denom = (np.abs(y_true)+np.abs(y_pred))/2
    return 100*np.mean(np.where(denom==0,0, np.abs(y_pred-y_true)/denom))


In [ ]:
import matplotlib.pyplot as plt

def plot_actual_vs_pred(dates, actual, preds, conf=None, interventions=None, title=''):
    plt.figure(figsize=(12,5))
    plt.plot(dates, actual, label='actual')
    plt.plot(dates, preds, label='pred')
    if conf is not None:
        plt.fill_between(dates, conf[:,0], conf[:,1], alpha=0.2)
    if interventions:
        for name, dates_list in interventions.items():
            for d in dates_list:
                plt.axvline(pd.to_datetime(d), linestyle='--', alpha=0.6)
    plt.legend(); plt.title(title); plt.xlabel('date')
    plt.show()
